### np.reshape

In [1]:
def get_shape(a):
    if not isinstance(a, list):
        return ()
    if len(a) == 0:
        return (0,)
    inner_shape = get_shape(a[0])
    for i in range(1, len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array detected: the shape of first row is {inner_shape}, "
                f"but row {i} has shape {get_shape(a[i])}"
            )
    return (len(a),) + inner_shape

def flatten(a):
    result = []
    if not isinstance(a, list):
        return [a]
    for item in a:
        result.extend(flatten(item))
    return result

In [2]:
def _build_from_shape(iterator, shape):
    # shape is a tuple of non-negative ints, no -1 here
    if len(shape) == 1:
        size = shape[0]
        row = []
        for _ in range(size):
            row.append(next(iterator))
        return row

    size = shape[0]
    rest = shape[1:]
    result = []
    for _ in range(size):
        result.append(_build_from_shape(iterator, rest))
    return result

def np_reshape(a, newshape):
    if not isinstance(a, list):
        raise TypeError(f"np_reshape expects a list, got {type(a).__name__}")

    original_shape = get_shape(a)
    flat = flatten(a)
    total = len(flat)

    # normalize newshape to tuple
    if isinstance(newshape, int):
        newshape = (newshape,)
    elif isinstance(newshape, list):
        newshape = tuple(newshape)
    elif not isinstance(newshape, tuple):
        raise TypeError("newshape must be int, tuple, or list")

    # validate newshape and handle -1
    inferred_index = None
    known_product = 1

    for i, dim in enumerate(newshape):
        if dim == -1:
            if inferred_index is not None:
                raise ValueError("only one dimension can be -1")
            inferred_index = i
        else:
            if not isinstance(dim, int):
                raise TypeError(f"dimension {dim} is not an integer")
            if dim < 0:
                raise ValueError(f"negative dimension {dim} is not allowed")
            known_product *= dim

    if inferred_index is not None:
        if known_product == 0:
            raise ValueError("cannot infer dimension with zero in newshape")
        if total % known_product != 0:
            raise ValueError("cannot infer dimension: size mismatch")
        inferred_dim = total // known_product
        newshape = list(newshape)
        newshape[inferred_index] = inferred_dim
        newshape = tuple(newshape)
    else:
        if known_product != total:
            raise ValueError(
                f"cannot reshape array of size {total} into shape {newshape}"
            )

    it = iter(flat)
    result = _build_from_shape(it, newshape)
    return result

### test_cases

In [4]:
print(np_reshape([1,2,3,4,5,6], (2,3)))
print(np_reshape([[1,2,3],[4,5,6]], (6,)))
print(np_reshape([1,2,3,4,5,6], (-1,2)))
print(np_reshape([[0,1,2,3],[4,5,6,7]], (2, 2, 2)))

[[1, 2, 3], [4, 5, 6]]
[1, 2, 3, 4, 5, 6]
[[1, 2], [3, 4], [5, 6]]
[[[0, 1], [2, 3]], [[4, 5], [6, 7]]]


In [5]:
print(np_reshape([1, 2, 3, 4], (3, 2)))

ValueError: cannot reshape array of size 4 into shape (3, 2)

In [6]:
np_reshape([1, 2, 3, 4], (-1, -1))

ValueError: only one dimension can be -1

In [7]:
np_reshape([1, 2, 3, 4, 5], (2, -1))

ValueError: cannot infer dimension: size mismatch

In [8]:
np_reshape([1, 2, 3, 4], (-2, 2))

ValueError: negative dimension -2 is not allowed

In [9]:
p_reshape([1, 2, 3, 4], ("2", 2))

NameError: name 'p_reshape' is not defined

In [10]:
np_reshape([1, 2, 3, 4], (2.5, 2))

TypeError: dimension 2.5 is not an integer

In [11]:
np_reshape(5, (1, 1))  

TypeError: np_reshape expects a list, got int

In [12]:
np_reshape([1, 2, 3, 4], None)

TypeError: newshape must be int, tuple, or list